# Beating Wall Street's Forecast Doesn't Lift Chip Stocks. Nvidia Reports Tonight.

**In the last 60 quarterly reports from five chip companies, the company beat Wall Street's forecast 92% of the time, and the stock still fell the next day after 55% of them. What moves these stocks is the outlook for next quarter, and the price investors put on it.**

Every quarter, analysts forecast a company's sales and profit. Their average forecast is the **estimate**. When the actual result comes in higher, the company **beat**. Nvidia reports its quarter after the market closes today, November 17, against an estimate of **$109.0 billion in sales**.

It reports into a falling market. Every major chip stock dropped on Monday. This notebook compares Nvidia with four companies that sell into the same AI data centers:

| Company | What it sells |
| --- | --- |
| **Nvidia** (NVDA) | AI chips, the processors that train and run AI models |
| **AMD** (AMD) | AI chips and server processors, Nvidia's closest rival |
| **Micron** (MU) | Memory, which sits beside every AI chip |
| **Broadcom** (AVGO) | Custom AI chips built for Google, Meta, and others |
| **TSMC** (TSM) | The factories that make chips for Nvidia, AMD, and Broadcom |

Each section heading states its finding, so the table of contents reads as the argument. All data comes from the [Financial Modeling Prep](https://site.financialmodelingprep.com/) API and Nvidia's own results releases. Use the buttons above to download the notebook or open it in Colab and rerun every number.

## Setup

```bash
pip install pandas numpy plotly requests scikit-learn
```

Analyst estimates and revenue by region need an FMP Starter plan or higher. Set your key as the `FMP_API_KEY` environment variable, or paste it when asked.

In [1]:
import os
import warnings
from getpass import getpass

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests
from sklearn.linear_model import LinearRegression

# the key stays out of the notebook: read it from the environment, or ask for it
API_KEY = os.environ.get("FMP_API_KEY") or getpass("FMP API key: ")
BASE = "https://financialmodelingprep.com/stable/"

def fmp(endpoint, **params):
    """Call one FMP endpoint and return the JSON."""
    response = requests.get(BASE + endpoint, params={**params, "apikey": API_KEY}, timeout=30)
    response.raise_for_status()
    return response.json()

# the five chip companies, in the order every chart and table uses
CHIPS = ["NVDA", "AMD", "MU", "AVGO", "TSM"]
BENCH = ["SMH", "QQQ"]
NAMES = {"NVDA": "Nvidia", "AMD": "AMD", "MU": "Micron", "AVGO": "Broadcom", "TSM": "TSMC",
         "SMH": "Chip ETF (SMH)", "QQQ": "Nasdaq 100 (QQQ)"}
# TSMC reports before the U.S. open, so its stock reacts the same day; the others report after the close
BEFORE_OPEN = {"TSM"}
# prices through Monday's close, the last full day before Nvidia reports
AS_OF = "2026-11-16"

# one fixed color per company, checked for colorblind separation on a dark background
COLORS = {"NVDA": "#0ea371", "AMD": "#e06410", "MU": "#0b8fd0", "AVGO": "#e0368a", "TSM": "#8457f0",
          "SMH": "#cbd5e1", "QQQ": "#64748b", "Down": "#f87171", "Up": "#34d399"}
pio.templates["post"] = go.layout.Template(
    layout=dict(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(family="Inter, system-ui, sans-serif", color="#cbd5e1", size=13),
        title=dict(font=dict(size=17, color="#f1f5f9"), x=0, xanchor="left"),
        xaxis=dict(gridcolor="rgba(148,163,184,0.15)", zerolinecolor="rgba(148,163,184,0.3)"),
        yaxis=dict(gridcolor="rgba(148,163,184,0.15)", zerolinecolor="rgba(148,163,184,0.3)"),
        colorway=[COLORS[t] for t in CHIPS],
        margin=dict(l=10, r=10, t=60, b=10),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(bgcolor="#0f172a", font=dict(color="#f1f5f9")),
    )
)
pio.templates.default = "plotly_dark+post"
# show charts in JupyterLab and VS Code, and keep their data in the saved notebook
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.options.display.float_format = "{:,.2f}".format

## 1. Every chip stock fell on Monday, not only Nvidia

Prices through Monday's close, the last full day before Nvidia reports.

In [2]:
def closes(symbol, start="2023-01-01"):
    data = pd.DataFrame(fmp("historical-price-eod/light", symbol=symbol, **{"from": start}))
    return data.assign(date=pd.to_datetime(data["date"])).set_index("date")["price"].sort_index()

prices = pd.DataFrame({t: closes(t) for t in CHIPS + BENCH}).sort_index().dropna().loc[:AS_OF]
quotes = pd.DataFrame([fmp("quote", symbol=t)[0] for t in CHIPS + BENCH]).set_index("symbol")

year = prices.loc[prices.index[-1] - pd.DateOffset(years=1):]
rout = pd.DataFrame({
    "close": prices.iloc[-1],
    "Monday": prices.iloc[-1] / prices.iloc[-2] - 1,
    "from 1-year high": prices.iloc[-1] / year.max() - 1,
}).rename(index=NAMES)
rout.style.format({"close": "${:,.2f}", "Monday": "{:+.1%}", "from 1-year high": "{:+.0%}"})

,close,Monday,from 1-year high
Nvidia,$156.14,-4.1%,-35%
AMD,$504.28,-5.6%,-22%
Micron,$719.02,-7.6%,-41%
Broadcom,$333.40,-3.9%,-31%
TSMC,$373.45,-6.3%,-23%
Chip ETF (SMH),$461.90,-5.1%,-31%
Nasdaq 100 (QQQ),$661.75,-2.7%,-13%


In [3]:
since = prices.loc["2026-09-30":]
indexed = since / since.iloc[0] - 1
rout_fig = go.Figure()
for t in BENCH + CHIPS:
    bench = t in BENCH
    rout_fig.add_trace(go.Scatter(
        x=indexed.index, y=indexed[t], name=NAMES[t], mode="lines",
        line=dict(color=COLORS[t], width=2, dash="dot" if bench else "solid"),
        hovertemplate=f"{NAMES[t]}: " + "%{y:+.1%}<extra></extra>",
    ))
# label each line at its end, nudged apart so labels for lines that finish close together don't overlap
ends = indexed.iloc[-1].sort_values(ascending=False)
label_y, gap = [], 0.035
for value in ends:
    label_y.append(min(value, label_y[-1] - gap) if label_y else value)
for (t, value), y in zip(ends.items(), label_y):
    rout_fig.add_annotation(x=indexed.index[-1], y=y, text=f"{t} {value:+.0%}", showarrow=False,
                            xanchor="left", xshift=6, font=dict(color="#e2e8f0", size=12))
rout_fig.update_layout(title="Every chip stock fell, not only Nvidia", yaxis_title="Change since Sep 30",
                       yaxis_tickformat="+.0%", hovermode="x unified", height=460,
                       margin=dict(r=90), legend=dict(orientation="h", y=-0.12))
rout_fig.show()

All five fell about 4% to 8% on Monday, the day the Commerce Department announced new limits on chip sales to China (section 5). Each now trades 22% to 41% below its high of the past year. The selloff hit the whole group, so Nvidia's report has to be read against its peers, not alone.

## 2. The businesses are booming: sales up 42% to 379% in a year

Before the stocks, the companies. First, sales growth against the same quarter a year earlier:

In [4]:
def earnings(symbol):
    data = pd.DataFrame(fmp("earnings", symbol=symbol, limit=20))
    data["date"] = pd.to_datetime(data["date"])
    return data.sort_values("date")

earn = {t: earnings(t) for t in CHIPS}
reported = {t: e.dropna(subset=["revenueActual"]).reset_index(drop=True) for t, e in earn.items()}

growth = pd.DataFrame({t: (e.set_index("date")["revenueActual"] / e.set_index("date")["revenueActual"].shift(4) - 1)
                       for t, e in reported.items()})
growth_long = growth.reset_index().melt(id_vars="date", var_name="symbol", value_name="growth").dropna()
growth_long = growth_long[growth_long["date"] >= "2024-01-01"]
growth_long["company"] = growth_long["symbol"].map(NAMES)

growth_fig = px.line(growth_long, x="date", y="growth", color="company", markers=True,
                     color_discrete_map={NAMES[t]: COLORS[t] for t in CHIPS},
                     category_orders={"company": [NAMES[t] for t in CHIPS]})
growth_fig.update_traces(line=dict(width=2), marker=dict(size=8),
                         hovertemplate="%{fullData.name}, reported %{x|%b %d, %Y}: revenue %{y:+.0%} vs. a year earlier<extra></extra>")
growth_fig.update_layout(title="All five grew sales at least 40% from a year earlier. Micron grew fastest", yaxis_tickformat="+.0%",
                         yaxis_title="", xaxis_title="Report date", height=460, legend_title_text="",
                         legend=dict(orientation="h", y=-0.18))
growth_fig.show()

Growth shows how fast sales rise. **Operating margin** shows how much of each sales dollar is left as profit after the costs of making the product, research, and running the company.

<img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/d/d4/NVIDIA_H100_%28%E6%9E%81%E5%AE%A2%E6%B9%BEGeekerwan%29_011.png/1280px-NVIDIA_H100_%28%E6%9E%81%E5%AE%A2%E6%B9%BEGeekerwan%29_011.png" alt="A row of Nvidia H100 data-center GPU cards on a white surface" width="100%">

*Nvidia H100 cards, the data-center chips behind its first surge in AI sales. Photo: 极客湾Geekerwan, [CC BY 3.0](https://creativecommons.org/licenses/by/3.0), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:NVIDIA_H100_(%E6%9E%81%E5%AE%A2%E6%B9%BEGeekerwan)_011.png).*

In [5]:
def income(symbol):
    data = pd.DataFrame(fmp("income-statement", symbol=symbol, period="quarter", limit=12))
    data["date"] = pd.to_datetime(data["date"])
    return data.sort_values("date").set_index("date")

inc = {t: income(t) for t in CHIPS}
margin_rows = []
for t, df in inc.items():
    for date, row in df.iterrows():
        margin_rows.append({"company": NAMES[t], "date": date,
                            "gross margin": row["grossProfit"] / row["revenue"],
                            "operating margin": row["operatingIncome"] / row["revenue"]})
margins = pd.DataFrame(margin_rows)

margin_fig = px.line(margins, x="date", y="operating margin", color="company", markers=True,
                     color_discrete_map={NAMES[t]: COLORS[t] for t in CHIPS},
                     category_orders={"company": [NAMES[t] for t in CHIPS]})
margin_fig.update_traces(line=dict(width=2), marker=dict(size=8),
                         hovertemplate="%{fullData.name}, quarter ended %{x|%b %Y}: %{y:.0%} operating margin<extra></extra>")
margin_fig.update_layout(title="Micron went from a loss to the highest profit margin in the group", yaxis_tickformat=".0%",
                         yaxis_title="", xaxis_title="Quarter ended", height=460, legend_title_text="",
                         legend=dict(orientation="h", y=-0.18))
margin_fig.show()

# the last four quarters together; margins are ratios, so TSMC's Taiwan dollars compare directly
ttm = pd.DataFrame({t: df.iloc[-4:][["revenue", "grossProfit", "operatingIncome"]].sum() for t, df in inc.items()}).T
summary = pd.DataFrame({
    "latest quarter revenue ($B)": {t: e["revenueActual"].iloc[-1] / 1e9 for t, e in reported.items()},
    "growth vs. a year earlier": growth.ffill().iloc[-1],
    "gross margin": ttm["grossProfit"] / ttm["revenue"],
    "operating margin": ttm["operatingIncome"] / ttm["revenue"],
}).rename(index=NAMES)
summary.style.format({"latest quarter revenue ($B)": "{:,.1f}", "growth vs. a year earlier": "{:+.0%}",
                      "gross margin": "{:.0%}", "operating margin": "{:.0%}"})

,latest quarter revenue ($B),growth vs. a year earlier,gross margin,operating margin
Nvidia,96.2,+106%,75%,65%
AMD,13.4,+45%,54%,18%
Micron,54.2,+379%,81%,75%
Broadcom,29.6,+86%,69%,48%
TSMC,46.1,+42%,66%,58%


These are some of the strongest businesses in the market. Nvidia's sales doubled in a year, to $96 billion a quarter, and it keeps 65 cents of operating profit on every dollar. Micron grew fastest, nearly fivefold, because AI chips need memory that is in short supply.

Note Micron's line in the margin chart. In late 2023 it was losing money. Over the last four quarters it kept 75 cents on the dollar, more than Nvidia. Memory prices rise and fall in cycles, and Micron's profits follow them. That matters twice more: in section 4, for Micron's price, and in section 6, because Micron's higher prices are Nvidia's higher costs.

## 3. Beating the forecast didn't lift the stock: 92% beat, 55% fell

For each of the last 12 reports from each company, 60 in all: did sales beat the estimate, and what did the stock do on the next trading day? (TSMC reports before the U.S. market opens, so its reaction is the same day.)

In [6]:
rows = []
for t, e in reported.items():
    days = prices.index
    for _, r in e[e["date"] >= "2023-03-01"].iterrows():
        after = days[days >= r["date"]] if t in BEFORE_OPEN else days[days > r["date"]]
        if after.empty:
            continue
        day = after[0]
        prev = days[days < day][-1]
        move = prices.loc[day, t] / prices.loc[prev, t] - 1
        sector = prices.loc[day, "SMH"] / prices.loc[prev, "SMH"] - 1
        rows.append({"symbol": t, "company": NAMES[t], "reported": r["date"], "reaction day": day,
                     "revenue surprise": r["revenueActual"] / r["revenueEstimated"] - 1,
                     "EPS surprise": r["epsActual"] / r["epsEstimated"] - 1,
                     "stock move": move, "chip ETF that day": sector, "move vs. chip ETF": move - sector})
reports = pd.DataFrame(rows).groupby("symbol").tail(12).reset_index(drop=True)

scorecard = reports.groupby("company", sort=False).agg(**{
    "reports": ("stock move", "size"),
    "beat revenue estimate": ("revenue surprise", lambda v: (v > 0).mean()),
    "stock fell next day": ("stock move", lambda v: (v < 0).mean()),
    "typical move, either way": ("stock move", lambda v: v.abs().mean()),
})
print(f"{len(reports)} reports. Revenue beat the estimate {(reports['revenue surprise'] > 0).mean():.0%} of the time "
      f"(EPS {(reports['EPS surprise'] > 0).mean():.0%}). The stock fell the next day after {(reports['stock move'] < 0).mean():.0%}.")
scorecard.style.format({"beat revenue estimate": "{:.0%}", "stock fell next day": "{:.0%}", "typical move, either way": "{:.1%}"})

60 reports. Revenue beat the estimate 92% of the time (EPS 92%). The stock fell the next day after 55%.


,reports,beat revenue estimate,stock fell next day,"typical move, either way"
company,,,,
Nvidia,12,100%,58%,5.6%
AMD,12,92%,58%,7.3%
Micron,12,83%,50%,8.8%
Broadcom,12,83%,50%,9.3%
TSMC,12,100%,58%,3.0%


In [7]:
X = reports[["revenue surprise"]]
y = reports["move vs. chip ETF"]
fit = LinearRegression().fit(X, y)
r2 = fit.score(X, y)
print(f"R² = {r2:.2f}: the size of the beat explains about {r2:.0%} of the next-day move.")

beat_fig = go.Figure()
for t in CHIPS:
    d = reports[reports["symbol"] == t]
    beat_fig.add_trace(go.Scatter(
        x=d["revenue surprise"], y=d["move vs. chip ETF"], mode="markers", name=NAMES[t],
        marker=dict(color=COLORS[t], size=10, line=dict(color="#0b1220", width=2)),
        customdata=np.stack([d["reported"].dt.strftime("%b %Y"), d["stock move"]], axis=1),
        hovertemplate=f"{NAMES[t]}, " + "%{customdata[0]}<br>Revenue beat %{x:+.1%}<br>"
                      "Stock %{customdata[1]:+.1%}, %{y:+.1%} vs. chip ETF<extra></extra>",
    ))
xs = np.linspace(reports["revenue surprise"].min(), reports["revenue surprise"].max(), 50)
beat_fig.add_trace(go.Scatter(x=xs, y=fit.predict(pd.DataFrame({"revenue surprise": xs})), mode="lines",
                              name=f"Best fit (R² {r2:.2f})", line=dict(color="#cbd5e1", width=2, dash="dash"),
                              hoverinfo="skip"))
beat_fig.add_hline(y=0, line_color="rgba(148,163,184,0.5)")
beat_fig.update_layout(title="A bigger beat didn't bring a bigger gain (each dot is one report)", xaxis_title="Revenue vs. analyst estimate",
                       yaxis_title="Next-day move vs. chip ETF", xaxis_tickformat="+.0%", yaxis_tickformat="+.0%",
                       height=480, legend=dict(orientation="h", y=-0.18))
beat_fig.show()

R² = 0.02: the size of the beat explains about 2% of the next-day move.


These companies beat the sales estimate 92% of the time, and the stock fell the next day 55% of the time, about as often as a coin flip.

The chart tests whether a *bigger* beat brings a bigger gain. Each dot is one report. The move is measured against the chip ETF (SMH), a fund that holds the whole sector, so a day when every chip stock rose or fell doesn't count for or against one company. The fitted line is nearly flat: the size of the beat explains about 2% of the move (R² = 0.02).

### Why a beat is expected: Nvidia sets the bar, then clears it by about 5%

Each quarter Nvidia publishes **guidance**, its own forecast for next quarter's sales. Analysts build their estimates from it. The chart compares the two with actual sales for the last five quarters and tonight. Guidance comes from Nvidia's results releases; estimates and actuals from FMP.

In [8]:
# Nvidia's revenue guidance (midpoint, plus or minus 2%), from the outlook in each prior quarter's results release
GUIDANCE = {"Q2 FY26": ("2025-08-27", 45.0), "Q3 FY26": ("2025-11-19", 54.0), "Q4 FY26": ("2026-02-25", 65.0),
            "Q1 FY27": ("2026-05-20", 78.0), "Q2 FY27": ("2026-08-26", 91.0), "Q3 FY27, tonight": ("2026-11-17", 108.0)}
nv_earn = earn["NVDA"].set_index("date")
guide = pd.DataFrame([{"quarter": q, "guidance": g,
                       "estimate": nv_earn.loc[d, "revenueEstimated"] / 1e9,
                       "actual": nv_earn.loc[d, "revenueActual"] / 1e9} for q, (d, g) in GUIDANCE.items()]).set_index("quarter")
guide["estimate vs. guidance"] = guide["estimate"] / guide["guidance"] - 1
guide["actual vs. guidance"] = guide["actual"] / guide["guidance"] - 1
done = guide.dropna(subset=["actual"])
print(f"Estimate above guidance: {done['estimate vs. guidance'].min():+.1%} to {done['estimate vs. guidance'].max():+.1%}. "
      f"Actual above guidance: {done['actual vs. guidance'].min():+.1%} to {done['actual vs. guidance'].max():+.1%}, "
      f"average {done['actual vs. guidance'].mean():+.1%}.")
expected = guide.loc["Q3 FY27, tonight", "guidance"] * (1 + done["actual vs. guidance"].mean())
print(f"Tonight: guidance $108.0B, estimate ${guide.iloc[-1]['estimate']:.1f}B. "
      f"At its average beat of guidance, sales would be about ${expected:.0f}B.")

guide_fig = go.Figure()
for col, color, name in [("guidance", "#64748b", "Nvidia's guidance"), ("estimate", "#cbd5e1", "Wall Street's estimate"),
                         ("actual", COLORS["NVDA"], "Actual")]:
    gap = guide[col] / guide["guidance"] - 1
    guide_fig.add_trace(go.Bar(x=guide.index, y=guide[col], name=name, marker_color=color,
                               marker_line_color="#0b1220", marker_line_width=2,
                               text=None if col == "guidance" else gap.map(lambda v: "" if pd.isna(v) else f"{v:+.0%}"),
                               textposition="outside", textfont=dict(color="#e2e8f0", size=11), cliponaxis=False,
                               customdata=gap,
                               hovertemplate=f"{name}: " + "$%{y:.1f}B (%{customdata:+.1%} vs. guidance)<extra></extra>"))
guide_fig.add_trace(go.Bar(x=["Q3 FY27, tonight"], y=[expected], name=f"At its usual beat (about ${expected:.0f}B)",
                           marker_color="rgba(14,163,113,0.25)", marker_line_color=COLORS["NVDA"], marker_line_width=2,
                           text=[f"{done['actual vs. guidance'].mean():+.0%}?"], textposition="outside",
                           textfont=dict(color="#e2e8f0", size=11), cliponaxis=False,
                           hovertemplate="If Nvidia beats its guidance by its average: $%{y:.0f}B<extra></extra>"))
guide_fig.update_layout(title="Nvidia guides low, analysts estimate 1% to 2% above it, and sales land about 5% above it",
                        barmode="group", yaxis_title="Quarterly sales ($B)", yaxis_tickprefix="$", height=460,
                        legend=dict(orientation="h", y=-0.15), bargap=0.25)
guide_fig.show()

Estimate above guidance: +1.2% to +2.3%. Actual above guidance: +3.9% to +5.7%, average +4.9%.
Tonight: guidance $108.0B, estimate $109.0B. At its average beat of guidance, sales would be about $113B.


The pattern repeats every quarter. Analysts set their estimate 1% to 2% above Nvidia's guidance, and Nvidia beats its guidance by 4% to 6%. A beat of the estimate is close to certain, so investors don't reward it.

Tonight's guidance is $108.0 billion and the estimate is $109.0 billion, only 0.9% above it. If Nvidia beats its guidance by its usual 5%, sales land near **$113 billion**. That, not $109 billion, is the number investors expect. A result near the estimate would be a miss in practice.

## 4. Nvidia costs 9.9 times next year's profit, less than any peer except Micron

If beats don't explain the stocks, the price investors pay for future profits might. The standard measure is the **price-to-earnings ratio (P/E)**: the share price divided by profit per share. A P/E of 10 means investors pay $10 for each $1 of yearly profit. This uses analysts' estimate of profit for each company's next full fiscal year, which ends between September 2027 and January 2028.

In [9]:
usd_twd = fmp("quote", symbol="USDTWD")[0]["price"]
today = pd.Timestamp(AS_OF)

def next_year_eps(symbol):
    """Consensus EPS for the next full fiscal year: the one ending 9 to 15 months from now."""
    est = pd.DataFrame(fmp("analyst-estimates", symbol=symbol, period="annual", limit=10))
    est["date"] = pd.to_datetime(est["date"])
    row = est[(est["date"] > today + pd.DateOffset(months=9)) & (est["date"] <= today + pd.DateOffset(months=15))].iloc[0]
    # TSMC's estimates come in Taiwan dollars per ADR; its stock trades in U.S. dollars
    return row["epsAvg"] / (usd_twd if symbol == "TSM" else 1), row["date"]

val_rows = []
for t in CHIPS:
    price = prices[t].iloc[-1]
    trailing = reported[t]["epsActual"].iloc[-4:].sum()
    forward, fy_end = next_year_eps(t)
    val_rows.append({"company": NAMES[t], "symbol": t, "price": price, "market value ($B)": quotes.loc[t, "marketCap"] / 1e9,
                     "P/E, last four quarters": price / trailing, "P/E, next year": price / forward,
                     "expected profit growth": forward / trailing - 1})
val = pd.DataFrame(val_rows).set_index("company")
val[["P/E, last four quarters", "P/E, next year", "expected profit growth"]].style.format(
    {"P/E, last four quarters": "{:.1f}", "P/E, next year": "{:.1f}", "expected profit growth": "{:+.0%}"})

,"P/E, last four quarters","P/E, next year",expected profit growth
company,,,
Nvidia,22.3,9.9,+125%
AMD,75.8,28.3,+168%
Micron,9.5,4.1,+133%
Broadcom,34.2,17.3,+98%
TSMC,24.2,16.7,+45%


In [10]:
val_fig = go.Figure()
for t in CHIPS:
    r = val[val["symbol"] == t].iloc[0]
    val_fig.add_trace(go.Scatter(
        x=[r["expected profit growth"]], y=[r["P/E, next year"]], mode="markers+text", name=NAMES[t],
        text=[NAMES[t]], textposition="top center", textfont=dict(color="#e2e8f0"),
        marker=dict(color=COLORS[t], size=np.sqrt(r["market value ($B)"]) * 0.8, line=dict(color="#0b1220", width=2)),
        hovertemplate=f"{NAMES[t]}<br>Next-year P/E " + "%{y:.1f}<br>Expected profit growth %{x:+.0%}<br>"
                      f"Market value ${r['market value ($B)']:,.0f}B<extra></extra>",
    ))
val_fig.update_layout(title="Nvidia is priced lowest for its expected growth, apart from Micron (bubble size = market value)",
                      xaxis_title="Next year's expected EPS vs. the last four quarters", yaxis_title="Price / next year's expected EPS",
                      xaxis_tickformat="+.0%", height=480, showlegend=False,
                      yaxis_range=[0, val["P/E, next year"].max() * 1.25])
val_fig.show()

TSMC, which makes Nvidia's chips, costs 16.7 times next year's profit, Broadcom 17.3, and AMD 28.3. Only Micron is cheaper, at 4.1, and section 2 showed why: investors expect its peak profits to fall when memory prices turn.

To see the size of the gap, apply each peer's P/E to Nvidia's expected profit:

In [11]:
nv_val = val.loc["Nvidia"]
forward_eps = nv_val["price"] / nv_val["P/E, next year"]
implied = pd.DataFrame({
    "next-year P/E": val["P/E, next year"],
    "Nvidia price at that P/E": val["P/E, next year"] * forward_eps,
}).drop(index="Nvidia")
implied["vs. Monday's close"] = implied["Nvidia price at that P/E"] / nv_val["price"] - 1
implied.loc["Peer median"] = [implied["next-year P/E"].median(), implied["next-year P/E"].median() * forward_eps,
                              implied["next-year P/E"].median() * forward_eps / nv_val["price"] - 1]
print(f"Nvidia: ${nv_val['price']:.2f}, {nv_val['P/E, next year']:.1f} times next year's expected EPS of ${forward_eps:.2f}")
implied.style.format({"next-year P/E": "{:.1f}", "Nvidia price at that P/E": "${:,.0f}", "vs. Monday's close": "{:+.0%}"})

Nvidia: $156.14, 9.9 times next year's expected EPS of $15.75


,next-year P/E,Nvidia price at that P/E,vs. Monday's close
company,,,
AMD,28.3,$446,+186%
Micron,4.1,$64,-59%
Broadcom,17.3,$272,+74%
TSMC,16.7,$264,+69%
Peer median,17.0,$268,+72%


At the peers' median P/E, Nvidia would trade near $268, 71% above Monday's close. At Micron's, it would trade near $64.

Investors are pricing Nvidia closer to a company whose profits they expect to shrink than to the companies around it. They aren't doubting the business in section 2. They are doubting the estimate, and the estimate isn't only analysts' optimism: in August, Nvidia [projected about 70% sales growth for fiscal 2028](https://www.implicator.ai/nvidia-70-percent-growth-forecast-margin-outlook/). The low P/E says the market doesn't fully believe the company's own outlook after the past two weeks.

## 5. China is a bigger risk for the toolmakers than for Nvidia

Three things happened in the past week:

- The Commerce Department [issued a rule to further limit chip shipments to China](https://www.wsj.com/world/asia/commerce-department-rule-aims-to-further-limit-chip-shipments-to-china-c6244c33).
- Treasury Secretary Scott Bessent [said Nvidia needs a "dramatic shakeup"](https://247wallst.com/investing/2026/11/16/bessent-says-nvidia-needs-a-dramatic-shakeup-as-a-singapore-firm-reportedly-moved-at-least-7-5-billion-of-nvidia-chips-to-china/) after a report that a Singapore company moved at least $7.5 billion of its chips into China.
- Nvidia's board, without founder Jensen Huang, who resigned as CEO last week, [formed a committee to oversee export rules](https://www.globenewswire.com/news-release/2026/11/17/3404362/0/en/nvidia-board-of-directors-establishes-national-security-and-export-control-oversight-committee.html).

How much depends on China? The map follows revenue by the region where each company bills its customers. On the left are the four companies that make the machines every chip factory needs: **ASML**, **Applied Materials**, **Lam Research**, and **KLA**. On the right are the chip companies. Broadcom doesn't report China separately, so it is left out. Hover over a line for its share.

In [12]:
def annual_revenue(symbol):
    """Reported revenue by fiscal year. Geographic segments don't always add up to it (TSMC leaves out Taiwan)."""
    return {int(r["fiscalYear"]): r["revenue"] for r in fmp("income-statement", symbol=symbol, period="annual", limit=5)}

TOOLS = ["ASML", "AMAT", "LRCX", "KLAC"]
TOOL_NAMES = {"ASML": "ASML", "AMAT": "Applied Materials", "LRCX": "Lam Research", "KLAC": "KLA"}
REGIONS = {"CHINA": "China", "HONG KONG": "China", "TAIWAN": "Taiwan", "TAIWAN, PROVINCE OF CHINA": "Taiwan",
           "KOREA, REPUBLIC OF": "Korea", "UNITED STATES": "United States", "NORTH AMERICA": "United States"}

def region_mix(symbol):
    """Share of the latest fiscal year's reported revenue billed to each region."""
    latest = fmp("revenue-geographic-segmentation", symbol=symbol)[0]
    total = annual_revenue(symbol).get(int(latest["fiscalYear"]), sum(latest["data"].values()))
    mix = {}
    for region, value in latest["data"].items():
        name = REGIONS.get(region.upper(), "Everywhere else")
        mix[name] = mix.get(name, 0) + value / total
    # whatever the segments leave out counts as everywhere else
    mix["Everywhere else"] = mix.get("Everywhere else", 0) + max(0, 1 - sum(mix.values()))
    return mix, f"FY{latest['fiscalYear']}"

# Broadcom reports Asia Pacific as one region, so it is left out
chain = {t: region_mix(t) for t in TOOLS + [t for t in CHIPS if t != "AVGO"]}

# a three-column map: toolmakers -> where their revenue is billed <- chipmakers' revenue by region
left = {t: (0, y) for t, y in zip(TOOLS, [3, 2, 1, 0])}
middle = {r: (1, y) for r, y in zip(["China", "Taiwan", "Korea", "United States"], [3, 2, 1, 0])}
sellers = [t for t in CHIPS if t != "AVGO"]
right = {t: (2, y) for t, y in zip(sellers, [3, 2, 1, 0])}

def curve(a, b, n=30):
    """A smooth S-curve between two points, as x and y lists."""
    s = np.linspace(0, 1, n)
    ease = 3 * s**2 - 2 * s**3
    return a[0] + (b[0] - a[0]) * s, a[1] + (b[1] - a[1]) * ease

map_fig = go.Figure()
for side, nodes in [("tools", left), ("sellers", right)]:
    for t, pos in nodes.items():
        mix, fy = chain[t]
        for region, rpos in middle.items():
            share = mix.get(region, 0)
            if share < 0.01:
                continue
            x, y = curve(pos, rpos)
            china = region == "China"
            who = TOOL_NAMES.get(t, NAMES.get(t))
            map_fig.add_trace(go.Scatter(
                x=x, y=y, mode="lines", hoverinfo="text",
                text=f"{who}: {share:.0%} of {fy} revenue billed to {region}",
                line=dict(width=1 + share * 40, color="rgba(245,158,11,0.75)" if china else "rgba(148,163,184,0.35)"),
                showlegend=False,
            ))
for nodes, label_side in [(left, "middle left"), (middle, "top center"), (right, "middle right")]:
    for name, (x, y) in nodes.items():
        color = COLORS.get(name, "#e2e8f0" if nodes is middle else "#94a3b8")
        label = TOOL_NAMES.get(name) or NAMES.get(name) or name
        map_fig.add_trace(go.Scatter(
            x=[x], y=[y], mode="markers+text", text=[label], textposition=label_side,
            textfont=dict(color="#f1f5f9", size=13), hoverinfo="skip", showlegend=False,
            marker=dict(size=22 if nodes is middle else 16, color=color, line=dict(color="#0b1220", width=2)),
        ))
for x, title in [(0, "Toolmakers sell to fabs in..."), (1, "Region"), (2, "...where chip revenue is billed")]:
    map_fig.add_annotation(x=x, y=3.65, text=f"<b>{title}</b>", showarrow=False, font=dict(color="#cbd5e1", size=12))
map_fig.update_layout(title="China buys more from the toolmakers than from the chipmakers (line width = share of revenue)",
                      height=520, xaxis=dict(visible=False, range=[-0.55, 2.55]), yaxis=dict(visible=False, range=[-0.5, 3.9]),
                      hovermode="closest", margin=dict(l=10, r=10, t=70, b=10))
map_fig.show()

The four toolmakers bill 29% to 34% of their revenue to China, as Chinese factories buy equipment to make their own chips. Nvidia bills 9%. AMD is the most exposed chip seller, at 22%.

For tonight, Nvidia's exposure is smaller still. Its guidance [assumes no data-center sales to China](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-second-quarter-fiscal-2027), so the new rule can't cut the $108 billion directly. The risk is indirect: scrutiny of where its chips end up, as in the Singapore report, and rules that could reach customers outside China.

## 6. Tonight's test is next quarter's forecast and margin, not the beat

<img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/e/ea/AI%E5%8A%A9%E6%89%8B%E8%83%BD%E5%B8%AE%E6%88%91%E8%B6%85%E9%A2%91%EF%BC%9FCOMPUTEX_NV%E5%B1%95%E5%8C%BA%E4%BD%93%E9%AA%8C_%28%E6%9E%81%E5%AE%A2%E6%B9%BEGeekerwan%29_09.png/1280px-AI%E5%8A%A9%E6%89%8B%E8%83%BD%E5%B8%AE%E6%88%91%E8%B6%85%E9%A2%91%EF%BC%9FCOMPUTEX_NV%E5%B1%95%E5%8C%BA%E4%BD%93%E9%AA%8C_%28%E6%9E%81%E5%AE%A2%E6%B9%BEGeekerwan%29_09.png" alt="A hand beside an open Nvidia server board with large GPU modules" width="100%">

*An Nvidia server board on display at Computex, beside a hand for scale. Most of tonight's sales come from systems built on boards like this. Photo: 极客湾Geekerwan, [CC BY 3.0](https://creativecommons.org/licenses/by/3.0), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:AI%E5%8A%A9%E6%89%8B%E8%83%BD%E5%B8%AE%E6%88%91%E8%B6%85%E9%A2%91%EF%BC%9FCOMPUTEX_NV%E5%B1%95%E5%8C%BA%E4%BD%93%E9%AA%8C_(%E6%9E%81%E5%AE%A2%E6%B9%BEGeekerwan)_09.png).*

Nvidia's record on the day after each report:

In [13]:
nv_reports = reports[reports["symbol"] == "NVDA"]
react_fig = go.Figure(go.Bar(
    x=nv_reports["reported"].dt.strftime("%b %Y"), y=nv_reports["stock move"],
    marker_color=np.where(nv_reports["stock move"] >= 0, COLORS["Up"], COLORS["Down"]),
    marker_line_color="#0b1220", marker_line_width=2,
    customdata=nv_reports["revenue surprise"],
    hovertemplate="%{x}: revenue beat %{customdata:+.1%}, stock %{y:+.1%} next day<extra></extra>",
))
for band in [0.08, -0.08]:
    react_fig.add_hline(y=band, line_dash="dot", line_color="#cbd5e1")
react_fig.add_annotation(x=1, xref="paper", y=-0.08, text="Options-implied move tonight, about ±8%", showarrow=False,
                         xanchor="right", yshift=-12, font=dict(color="#e2e8f0"))
react_fig.update_layout(title="Nvidia beat estimates in all 12 reports. The stock fell the next day after 7",
                        yaxis_tickformat="+.0%", yaxis_title="", xaxis_title="Report", height=440, showlegend=False)
react_fig.show()

Options traders are pricing a move of about 8% either way tomorrow, [a swing near $335 billion in market value](https://www.reuters.com/business/nvidia-shares-set-335-billion-swing-after-earnings-options-show-2026-11-17/). Sections 3 through 5 point to what will decide the direction:

1. **Sales near $113 billion, not $109 billion.** Section 3 showed that is the result investors already expect.
2. **Guidance for next quarter.** It is the new bar, and the only part of tonight's report that investors don't already expect.
3. **Gross margin**, the share of sales left after the cost of making the chips. Nvidia guided 74.0% for this quarter and has already said margins will [dip to 71% to 72% next quarter](https://www.implicator.ai/nvidia-70-percent-growth-forecast-margin-outlook/) because memory prices are rising. That is Micron's gain from section 2 showing up as Nvidia's cost. A deeper dip would hit the profit estimates behind section 4's P/E.

The next tests for the group come soon after: Broadcom reports on December 9 and Micron on December 23.

## Bottom line: a top business priced for doubt, and the forecast decides

Nvidia is one of the fastest-growing, most profitable companies in the market, priced as if its profits might fall. Whether that price is an opportunity depends on whether tonight's guidance and margins support the estimates. The beat itself, based on 60 reports, will say little.

*This is analysis of public data, not advice to buy or sell anything. Only invest money you can afford to lose.*

### Limits

- Estimates are FMP's averages. Some were set before the past two weeks.
- Fiscal years end in different months, so "next year" covers slightly different periods.
- Revenue by region follows billing, not where chips are used.
- Twelve reports per company is a small sample. The pattern holds across the group, but any one company's record can change.

### Sources

1. Financial Modeling Prep, [stable API](https://site.financialmodelingprep.com/developer/docs): prices, earnings and estimates, income statements, analyst estimates, revenue by region, and the USD/TWD exchange rate, pulled Nov 17, 2026.
2. NVIDIA, results releases with next-quarter outlook: [Q1 FY2026](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-first-quarter-fiscal-2026), [Q2 FY2026](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-second-quarter-fiscal-2026), [Q3 FY2026](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-third-quarter-fiscal-2026), [Q4 FY2026](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-fourth-quarter-and-fiscal-2026), [Q1 FY2027](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-first-quarter-fiscal-2027), [Q2 FY2027](https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-second-quarter-fiscal-2027).
3. Implicator, [coverage of Nvidia's second-quarter fiscal 2027 earnings call](https://www.implicator.ai/nvidia-70-percent-growth-forecast-margin-outlook/): the 70% sales growth outlook for fiscal 2028 and the gross-margin guidance, Aug 26, 2026.
4. Reuters, [Nvidia shares set for $335 billion swing after earnings, options show](https://www.reuters.com/business/nvidia-shares-set-335-billion-swing-after-earnings-options-show-2026-11-17/), Nov 17, 2026.
5. Wall Street Journal, [Commerce Department rule aims to further limit chip shipments to China](https://www.wsj.com/world/asia/commerce-department-rule-aims-to-further-limit-chip-shipments-to-china-c6244c33), Nov 16, 2026.
6. 24/7 Wall St., [Bessent says Nvidia needs a "dramatic shakeup"](https://247wallst.com/investing/2026/11/16/bessent-says-nvidia-needs-a-dramatic-shakeup-as-a-singapore-firm-reportedly-moved-at-least-7-5-billion-of-nvidia-chips-to-china/), Nov 16, 2026.
7. NVIDIA, [Board of Directors establishes National Security and Export Control Oversight Committee](https://www.globenewswire.com/news-release/2026/11/17/3404362/0/en/nvidia-board-of-directors-establishes-national-security-and-export-control-oversight-committee.html), Nov 17, 2026.
8. Damodaran, A. (2012). *Investment Valuation*, 3rd ed. Wiley. On valuing companies with multiples and comparables.